# N-CMAPSS DS05 — expanded search, three arms

## Why this run exists

v8 established the thing we had failed to establish anywhere in this project: on the matched
protocol the search objective is **discriminating**. Spearman(validation, test) = +0.822 over 120
trials, against +0.276 on the old representation. And the two quantities are related almost
deterministically:

    test_capped = 0.885 x val + 1.352      r = 0.980, n = 120

So validation gains convert to test gains nearly one-for-one here, and search effort is worth
spending. Inverting that line, reaching MSTSDN's 3.95 needs validation around **2.94**; our best in
120 trials was 3.528. That is roughly the same distance again as the whole of v8 achieved.

## What v8's trials say about where to look

Comparing the top 12 trials against the other 108, in normalised position within each range:

| parameter | top-12 | rest | p | |
|---|---|---|---|---|
| kernel | 0.188 | 0.426 | 0.0027 | **pinned at the low edge** |
| dense1 | 0.119 | 0.398 | 0.0006 | **pinned at the low edge** |
| batch | 0.292 | 0.468 | 0.0248 | interior, leaning small |

Everything else was interior and non-significant. Two parameters are pressing against the floor of
their range, so more evaluations inside the same box cannot reach the optimum — the box has to move.
This notebook widens it in exactly those directions and nowhere else, so the change is a response to
evidence rather than a general loosening.

Extending kernel below 2 means **kernel = 1**, a pointwise convolution. That is Peng's FCL trick, the
same one that fixed our FD003 architecture. We already know it works on another dataset; we simply
never let the search reach it here.

## Three arms at equal budget

v8's BO reached 3.833 across 46 GP-proposed points, while its own 14 random initialisation points had
already reached 3.621. The GP never beat its own initialisation. PSO, by contrast, was still
improving at the last evaluation (3.657 -> 3.528 over the final 14).

That makes **random search the honest control**, and this notebook runs all three at identical budget
on an identical space with an identical evaluator. Whichever wins, the comparison is reportable in
its own right — "BO did not beat random search at this budget on a 13-dimensional mixed space" is a
finding, not a failure, provided it is measured fairly.

## Discipline carried forward

- One budget everywhere; search and final train identically.
- Test recorded per trial, **never** returned to any optimiser.
- All three arms start far from every known-good point; the baseline and the two v8 winners are
  injected into each population afterwards, so no arm can regress below what we already have.
- Two-stage selection, and this time the stage-2 guard uses a **paired t-test on per-seed
  differences** rather than the 2-SE normal threshold v8 used, which was too lenient at n=5 and
  turned a p=0.086 result into an "ADOPT".
- Both target conventions reported throughout.

In [ ]:
import os, json, gc, time, random, math, warnings
os.environ['PYTHONHASHSEED']='42'
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, h5py
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel, WhiteKernel
from scipy.stats import norm, spearmanr, ttest_rel, mannwhitneyu
from scipy.stats.qmc import Sobol
import tensorflow as tf
for _g in tf.config.list_physical_devices('GPU'):
    try: tf.config.experimental.set_memory_growth(_g,True)
    except RuntimeError as _e: print('memory growth:',_e)
from tensorflow.keras import backend as K, regularizers
from tensorflow.keras.layers import (Input, Conv1D, MaxPooling1D, Dropout, Dense, LSTM,
                                     Concatenate, GlobalAveragePooling1D)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
def set_seeds(s):
    os.environ['PYTHONHASHSEED']=str(s); random.seed(s); np.random.seed(s)
    tf.keras.utils.set_random_seed(s)
def preflight():
    if not tf.config.list_physical_devices('GPU'): print('CPU mode'); return False
    try:
        x=np.random.rand(8,10,3).astype('float32'); y=np.random.rand(8,1).astype('float32')
        i=Input(shape=(10,3)); o=Dense(1)(GlobalAveragePooling1D()(Conv1D(4,3,padding='same')(i)))
        m=Model(i,o); m.compile('adam','mse'); m.fit(x,y,epochs=1,verbose=0); K.clear_session()
        print('GPU preflight passed'); return True
    except Exception as e:
        print('GPU preflight FAILED (%s)'%type(e).__name__); K.clear_session(); return False
USE_GPU=preflight(); DEVICE='/GPU:0' if USE_GPU else '/CPU:0'
try: tf.config.experimental.enable_op_determinism()
except Exception as e: print('determinism:',e)
print('TF',tf.__version__,'| device',DEVICE)

## Data — identical to v7/v8, so every number stays comparable

In [ ]:
SEED=42; SUB=50; VAL_UNITS=[5,6]; CAP=55; T=30
EPOCHS=100; PATIENCE=15
N_ARM=45; SHORTLIST=5; N_SEEDS2=6
set_seeds(SEED)
def read_sub(dset,step,dtype=np.float32,chunk=400_000):
    n=dset.shape[0]; out=[]
    for a in range(0,n,chunk):
        b=min(a+chunk,n); blk=dset[a:b]; first=(-a)%step
        if first<len(blk): out.append(blk[first::step].astype(dtype,copy=False))
    return np.concatenate(out)
with h5py.File('N-CMAPSS_DS05.h5','r') as h:
    Sd=read_sub(h['X_s_dev'],SUB); Vd=read_sub(h['X_v_dev'],SUB)
    Ad=read_sub(h['A_dev'],SUB);   Yd=read_sub(h['Y_dev'],SUB)
    St=read_sub(h['X_s_test'],SUB);Vt=read_sub(h['X_v_test'],SUB)
    At=read_sub(h['A_test'],SUB);  Yt=read_sub(h['Y_test'],SUB)
Fd=np.hstack([Sd,Vd]); Ft=np.hstack([St,Vt])
ud,cd,yd=Ad[:,0].astype(int),Ad[:,1].astype(int),Yd.ravel().astype(np.float32)
ut,ct,yt=At[:,0].astype(int),At[:,1].astype(int),Yt.ravel().astype(np.float32)
del Sd,Vd,Ad,Yd,St,Vt,At,Yt; gc.collect()
def windows(F,u,c,y,mask,T):
    uu,cc,FF,yy=u[mask],c[mask],F[mask],y[mask]
    key=uu.astype(np.int64)*100000+cc; o=np.argsort(key,kind='stable')
    FF,yy,key=FF[o],yy[o],key[o]; bnd=np.flatnonzero(np.diff(key))+1
    X,Y=[],[]
    for a,b in zip(np.r_[0,bnd],np.r_[bnd,len(key)]):
        A=FF[a:b]; L=yy[a:b]
        for e in range(T,len(A)+1): X.append(A[e-T:e]); Y.append(L[e-1])
    return np.asarray(X,np.float32),np.asarray(Y,np.float32)
tr=~np.isin(ud,VAL_UNITS); va=np.isin(ud,VAL_UNITS)
sc=MinMaxScaler().fit(Fd[tr])
Fd2=sc.transform(Fd).astype(np.float32); Ft2=sc.transform(Ft).astype(np.float32)
Xtr,ytr_raw=windows(Fd2,ud,cd,yd,tr,T)
Xva,yva_raw=windows(Fd2,ud,cd,yd,va,T)
Xte,yte_raw=windows(Ft2,ut,ct,yt,np.ones(len(yt),bool),T)
ytr=np.minimum(ytr_raw,CAP); yva=np.minimum(yva_raw,CAP); yte_cap=np.minimum(yte_raw,CAP)
del Fd,Ft,Fd2,Ft2; gc.collect()
SHAPE=(T,Xtr.shape[2])
triv_raw=float(np.sqrt(mean_squared_error(yte_raw,np.full_like(yte_raw,ytr_raw.mean()))))
triv_cap=float(np.sqrt(mean_squared_error(yte_cap,np.full_like(yte_cap,ytr.mean()))))
print('shape %s | train %s | val %s | test %s'%(SHAPE,format(len(ytr),','),
      format(len(yva),','),format(len(yte_raw),',')))
print('trivial floors -> raw %.3f | capped-%d %.3f'%(triv_raw,CAP,triv_cap))
print('leakage check: val units %s vs train units %s | overlap %s'
      %(VAL_UNITS,sorted(set(ud[tr])),sorted(set(ud[tr])&set(VAL_UNITS))))

## The expanded space

Four ranges widened, each because v8's trials pushed against that wall. Nine left exactly as they
were, because v8 found their optima in the interior.

In [ ]:
SPACE=[('n_conv','int',2,4), ('f1','int',24,192), ('f_growth','float',0.5,2.0),
       ('kernel','cat',[1,2,3,5,7]),          # was [2,3,5,7]  -- pinned low, p=0.0027
       ('use_pool','cat',[0,1]),
       ('n_lstm','int',1,2), ('lstm_units','int',24,320),   # was 24-192
       ('dropout','float',0.0,0.40), ('l2reg','log',1e-7,1e-1),
       ('dense1','int',8,256),                # was 32-256   -- pinned low, p=0.0006
       ('dense2','int',0,256),                # was 0-160
       ('lr','log',1e-4,5e-3), ('batch','cat',[64,128,256,512,1024])]  # was [128..1024]
D=len(SPACE); HPN=[s[0] for s in SPACE]
def decode(z):
    z=np.clip(np.asarray(z,float),0,1); out={}
    for zi,s in zip(z,SPACE):
        n,k=s[0],s[1]
        if k=='int':    out[n]=int(round(s[2]+zi*(s[3]-s[2])))
        elif k=='float':out[n]=float(s[2]+zi*(s[3]-s[2]))
        elif k=='log':  out[n]=float(np.exp(np.log(s[2])+zi*(np.log(s[3])-np.log(s[2]))))
        else:           out[n]=s[2][min(int(zi*len(s[2])),len(s[2])-1)]
    return out
def encode(kw):
    z=[]
    for s in SPACE:
        n,k=s[0],s[1]; v=kw[n]
        if k in ('int','float'): z.append((v-s[2])/(s[3]-s[2]))
        elif k=='log':           z.append((np.log(v)-np.log(s[2]))/(np.log(s[3])-np.log(s[2])))
        else:                    z.append((s[2].index(v)+0.5)/len(s[2]))
    return np.clip(np.array(z,float),0,1)
def nrmsd(a,b): return float(np.linalg.norm(np.asarray(a)-np.asarray(b))/np.sqrt(D))

BASE   =dict(n_conv=4,f1=96,f_growth=1.15,kernel=3,use_pool=1,n_lstm=1,lstm_units=80,
             dropout=0.10,l2reg=1e-7,dense1=192,dense2=96,lr=1e-3,batch=512)
V8_C2  =dict(n_conv=3,f1=131,f_growth=1.783738,kernel=5,use_pool=0,n_lstm=1,lstm_units=165,
             dropout=0.082654,l2reg=2.399954e-05,dense1=64,dense2=23,lr=0.000502,batch=1024)
V8_C5  =dict(n_conv=3,f1=68,f_growth=1.429117,kernel=2,use_pool=0,n_lstm=2,lstm_units=97,
             dropout=0.011585,l2reg=5.144132e-05,dense1=32,dense2=96,lr=0.000990,batch=128)
KNOWN=[('baseline',BASE),('v8_cand2',V8_C2),('v8_cand5',V8_C5)]
Z_KNOWN=[encode(k[1]) for k in KNOWN]
for nm,kw in KNOWN:
    d=decode(encode(kw))
    bad=[k for k in kw if d[k]!=kw[k] and abs(d[k]-kw[k])>1e-4*max(1,abs(kw[k]))]
    print('%-10s round-trip %s'%(nm,'OK' if not bad else 'MISMATCH '+str(bad)))
print('search dimension:',D)

In [ ]:
def build(kw,shape=SHAPE,seed=SEED):
    reg=regularizers.l2(kw['l2reg']) if kw['l2reg']>1e-6 else None
    kk=dict(kernel_initializer=tf.keras.initializers.GlorotUniform(seed=seed),kernel_regularizer=reg)
    inp=Input(shape=shape); c=inp
    for i in range(kw['n_conv']):
        f=max(8,int(round(kw['f1']*(kw['f_growth']**i))))
        c=Conv1D(f,kw['kernel'],padding='same',activation='relu',**kk)(c)
        if kw['dropout']>0: c=Dropout(kw['dropout'],seed=seed)(c)
        if kw['use_pool'] and i<2 and int(c.shape[1])>2: c=MaxPooling1D(2,padding='same')(c)
    cv=GlobalAveragePooling1D()(c)
    r=inp
    for j in range(kw['n_lstm']):
        r=LSTM(kw['lstm_units'],return_sequences=(j<kw['n_lstm']-1),**kk)(r)
    h=Concatenate()([cv,r])
    if kw['dense1']>=8: h=Dense(kw['dense1'],activation='relu',**kk)(h)
    if kw['dense2']>=16: h=Dense(kw['dense2'],activation='relu',**kk)(h)
    m=Model(inp,Dense(1,activation='linear',**kk)(h))
    m.compile(optimizer=tf.keras.optimizers.Adam(kw['lr']),loss='mse'); return m

TRIALS=[]
def evaluate(kw,seed=SEED,tag=''):
    """Returns VALIDATION only. Test is recorded for analysis and never fed back."""
    K.clear_session(); gc.collect(); set_seeds(seed); t0=time.time()
    try:
        with tf.device(DEVICE):
            m=build(kw,seed=seed)
            m.fit(Xtr,ytr,validation_data=(Xva,yva),epochs=EPOCHS,batch_size=kw['batch'],verbose=0,
                  callbacks=[EarlyStopping('val_loss',patience=PATIENCE,restore_best_weights=True),
                             ReduceLROnPlateau('val_loss',factor=0.5,patience=PATIENCE//2,min_lr=1e-6)])
            pv=m.predict(Xva,batch_size=8192,verbose=0).ravel()
            pt=m.predict(Xte,batch_size=8192,verbose=0).ravel()
        val=float(np.sqrt(mean_squared_error(yva,pv)))
        tc =float(np.sqrt(mean_squared_error(yte_cap,np.clip(pt,0,CAP))))
        trw=float(np.sqrt(mean_squared_error(yte_raw,pt)))
        npm=int(m.count_params())
    except Exception as e:
        print('   trial failed (%s) -> penalised'%type(e).__name__); val,tc,trw,npm=1e3,np.nan,np.nan,0
    if not np.isfinite(val): val=1e3
    TRIALS.append(dict(tag=tag,seed=seed,val=val,test_capped=tc,test_raw=trw,
                       params=npm,secs=round(time.time()-t0,1),**kw))
    return val

def far_design(n,refs,min_d=0.35,seed=SEED):
    eng=Sobol(d=D,scramble=True,seed=seed); pts=[]; guard=0
    while len(pts)<n and guard<50:
        guard+=1
        for c in eng.random(max(8,n*4)):
            if min(nrmsd(c,r) for r in refs)>=min_d:
                pts.append(c)
                if len(pts)==n: break
    rs=np.random.RandomState(seed)
    while len(pts)<n:
        c=rs.rand(D)
        if min(nrmsd(c,r) for r in refs)>=min_d: pts.append(c)
    return np.array(pts[:n])

print('warm-up: the three known-good configs, one seed each')
KNOWN_Y={}; KNOWN_REC={}
for nm,kw in KNOWN:
    KNOWN_Y[nm]=evaluate(kw,SEED,'known:'+nm); r=TRIALS[-1]; KNOWN_REC[nm]=dict(r)
    print('  %-10s val %6.3f | capped %6.3f | raw %7.3f | %8d params | %4.0fs'
          %(nm,r['val'],r['test_capped'],r['test_raw'],r['params'],r['secs']))
_s=np.mean([t['secs'] for t in TRIALS])
print('\nestimated: %.1f h for %d search evaluations + %.0f min for stage 2'
      %(3*N_ARM*_s/3600,3*N_ARM,(SHORTLIST+1)*N_SEEDS2*_s/60))

### Arm 1 — random search (Sobol)

The control. No model, no memory: just draws from the same box under the same far-start rule.

In [ ]:
N_INIT=15
Zr=far_design(N_ARM,Z_KNOWN,seed=SEED+1)
print('random arm: %d points, min distance to any known-good config %.3f'
      %(len(Zr),min(min(nrmsd(z,r) for r in Z_KNOWN) for z in Zr)))
t0=time.time(); RS_Y=[]
for i,z in enumerate(Zr):
    y=evaluate(decode(z),SEED,'random'); RS_Y.append(y)
    if (i+1)%5==0 or i==0: print('  random %2d/%d  val %7.3f  (best %7.3f)'%(i+1,N_ARM,y,min(RS_Y)))
RS_Z=Zr
print('random search done in %.1f min | best val %.3f'%((time.time()-t0)/60,min(RS_Y)))

### Arm 2 — Bayesian optimisation

Same GP as v8: Matern 5/2 with ARD, white-noise term, Expected Improvement. Its initial design is the
first 15 points of the same far-start rule, and the three known-good configs join the observed set
afterwards — in the population, not seeding it.

In [ ]:
Zobs=list(far_design(N_INIT,Z_KNOWN,seed=SEED+2)); Yobs=[]
for i,z in enumerate(Zobs):
    Yobs.append(evaluate(decode(z),SEED,'bo_init'))
    if (i+1)%5==0: print('  bo init %2d/%d  best %7.3f'%(i+1,N_INIT,min(Yobs)))
for (nm,kw),zk in zip(KNOWN,Z_KNOWN): Zobs.append(zk.copy()); Yobs.append(KNOWN_Y[nm])
print('  injected %d known-good configs into the GP population'%len(KNOWN))
def ei(mu,sd,best,xi=0.01):
    sd=np.maximum(sd,1e-9); imp=best-mu-xi; z=imp/sd
    return imp*norm.cdf(z)+sd*norm.pdf(z)
t0=time.time()
for it in range(N_ARM-N_INIT):
    Za,Ya=np.array(Zobs),np.array(Yobs); ok=Ya<999
    Zf,Yf=(Za[ok],Ya[ok]) if ok.sum()>=5 else (Za,Ya)
    gp=GaussianProcessRegressor(ConstantKernel(1.0)*Matern(length_scale=np.ones(D),nu=2.5)
                                +WhiteKernel(1e-2),normalize_y=True,n_restarts_optimizer=3,
                                random_state=SEED)
    gp.fit(Zf,Yf)
    cand=np.random.RandomState(SEED+it).rand(4000,D)
    mu,sd=gp.predict(cand,return_std=True)
    zb=cand[int(np.argmax(ei(mu,sd,Yf.min())))]
    y=evaluate(decode(zb),SEED,'bo'); Zobs.append(zb); Yobs.append(y)
    if (it+1)%5==0 or it==0: print('  bo %2d/%d  val %7.3f  (best %7.3f)'%(it+1,N_ARM-N_INIT,y,min(Yobs)))
print('BO done in %.1f min | best val %.3f'%((time.time()-t0)/60,min(Yobs)))
BO_Z,BO_Y=np.array(Zobs),np.array(Yobs)

### Arm 3 — particle swarm

The arm that was still improving when v8 ran out of budget. Particles 0, 1 and 2 are the baseline and
the two v8 winners; the remainder start far away with outward velocities.

In [ ]:
SWARM=9; ITERS=N_ARM//SWARM   # 5 iterations beats 3 wide ones
rng=np.random.RandomState(SEED+7)
P=far_design(SWARM,Z_KNOWN,seed=SEED+7)
for i,zk in enumerate(Z_KNOWN): P[i]=zk.copy()
V=(rng.rand(SWARM,D)-0.5)*0.30
pbest=P.copy(); pbest_y=np.full(SWARM,np.inf); gbest=None; gbest_y=np.inf
w_hi,w_lo,c1,c2=0.85,0.35,1.5,1.5
t0=time.time()
for it in range(ITERS):
    w_in=w_hi-(w_hi-w_lo)*it/max(1,ITERS-1)
    for k in range(SWARM):
        if it==0 and k<len(KNOWN):
            nm=KNOWN[k][0]; y=KNOWN_Y[nm]
            rec=dict(KNOWN_REC[nm]); rec['tag']='pso'; TRIALS.append(rec)
        else:
            y=evaluate(decode(P[k]),SEED,'pso')
        if y<pbest_y[k]: pbest_y[k]=y; pbest[k]=P[k].copy()
        if y<gbest_y:    gbest_y=y;    gbest=P[k].copy()
    r1,r2=rng.rand(SWARM,D),rng.rand(SWARM,D)
    V=np.clip(w_in*V+c1*r1*(pbest-P)+c2*r2*(gbest-P),-0.25,0.25); P=np.clip(P+V,0,1)
    print('  pso iter %d/%d  gbest %7.3f  spread %.3f'%(it+1,ITERS,gbest_y,float(np.mean(np.std(P,axis=0)))))
print('PSO done in %.1f min | best val %.3f'%((time.time()-t0)/60,gbest_y))

## Arm comparison at equal budget

In [ ]:
ALL=pd.DataFrame(TRIALS)
SR=ALL[ALL.tag.isin(['random','bo','bo_init','pso'])].copy()
SR['arm']=SR.tag.replace({'bo_init':'BO','bo':'BO','random':'RANDOM','pso':'PSO'})
print('%-8s %6s %8s %8s %8s %10s'%('arm','n','best val','median','best test','best params'))
for a in ['RANDOM','BO','PSO']:
    d=SR[(SR.arm==a)&(SR.val<999)]
    b=d.nsmallest(1,'val').iloc[0]
    print('%-8s %6d %8.3f %8.3f %8.3f %10d'%(a,len(d),d.val.min(),d.val.median(),b.test_capped,b.params))
print('\nbest-so-far validation by evaluation number:')
for a in ['RANDOM','BO','PSO']:
    d=SR[(SR.arm==a)].reset_index(drop=True); bs=np.minimum.accumulate(d.val.values)
    qs=[0,len(bs)//4,len(bs)//2,3*len(bs)//4,len(bs)-1]
    print('  %-8s'%a,' -> '.join('%d:%.3f'%(q+1,bs[q]) for q in qs))
print('\nv8 for reference: BO 3.833 after its init reached 3.621 | PSO 3.528 | baseline 4.335')

## Stage 2 — re-decide on fresh seeds, with a paired test this time

In [ ]:
short=SR[SR.val<999].nsmallest(SHORTLIST*3,'val').drop_duplicates(subset=HPN).head(SHORTLIST)
print('shortlist by stage-1 validation:')
print(short[['arm','val','test_capped','test_raw','params']].to_string(index=False))
def kw_of(row):
    get=(lambda n: getattr(row,n)) if hasattr(row,'_fields') else (lambda n: row[n])
    out={}
    for s in SPACE:
        n,k=s[0],s[1]; v=get(n)
        out[n]=int(v) if (k=='int' or n in ('kernel','use_pool','batch')) else float(v)
    return out
SEEDS2=[SEED+200+s for s in range(N_SEEDS2)]
PER={}; STAGE2=[]
cands=[('baseline',BASE),('v8_cand5',V8_C5)]+[('new%d_%s'%(i+1,r.arm),kw_of(r))
                                              for i,r in enumerate(short.itertuples())]
for name,kw in cands:
    v,c,rw=[],[],[]
    for s in SEEDS2:
        evaluate(kw,s,'stage2:'+name); t=TRIALS[-1]
        v.append(t['val']); c.append(t['test_capped']); rw.append(t['test_raw'])
    PER[name]=dict(val=np.array(v),capped=np.array(c),raw=np.array(rw))
    STAGE2.append(dict(name=name,val=np.mean(v),val_sd=np.std(v,ddof=1),
                       capped=np.mean(c),capped_sd=np.std(c,ddof=1),
                       raw=np.mean(rw),raw_sd=np.std(rw,ddof=1),**kw))
    print('  %-14s val %6.3f+/-%.3f | capped %6.3f+/-%.3f | raw %7.3f+/-%.3f'
          %(name,np.mean(v),np.std(v,ddof=1),np.mean(c),np.std(c,ddof=1),np.mean(rw),np.std(rw,ddof=1)))
S2=pd.DataFrame(STAGE2)

In [ ]:
win=S2[~S2['name'].isin(['baseline'])].nsmallest(1,'val').iloc[0]['name']
print('validation selects: %s   (selection never saw a test score)'%win)
print()
print('%-14s %16s %16s'%('','capped-55','raw'))
for nm in ['baseline','v8_cand5',win]:
    r=S2[S2['name']==nm].iloc[0]
    print('%-14s %8.3f+/-%.3f %8.3f+/-%.3f'%(nm,r.capped,r.capped_sd,r.raw,r.raw_sd))
print()
print('paired t-tests on the same %d seeds (this is the guard v8 got wrong):'%N_SEEDS2)
for ref in ['baseline','v8_cand5']:
    d=PER[win]['capped']-PER[ref]['capped']
    t,p=ttest_rel(PER[win]['capped'],PER[ref]['capped'])
    print('  %-9s vs %-12s  mean diff %+7.3f  paired SE %.3f  t %+6.2f  p %.4f %s'
          %(win,ref,d.mean(),d.std(ddof=1)/math.sqrt(len(d)),t,p,'  SIGNIFICANT' if p<0.05 else ''))
d_base=PER[win]['capped']-PER['baseline']['capped']
_,p_base=ttest_rel(PER[win]['capped'],PER['baseline']['capped'])
verdict='ADOPT' if (d_base.mean()<0 and p_base<0.05) else 'NOT SIGNIFICANT — keep the incumbent'
print('\nverdict:',verdict)

## Is the box still binding? — and how much room is left

In [ ]:
A=SR[(SR.val<999)].dropna(subset=['test_capped'])
rho,p=spearmanr(A.val,A.test_capped)
sl,ic=np.polyfit(A.val,A.test_capped,1)
print('Spearman(val, test-capped) over %d trials: %+.3f (p=%.3g)   [v8: +0.822]'%(len(A),rho,p))
print('test_capped = %.3f * val + %.3f    -> 3.95 needs val ~%.3f, best found %.3f'
      %(sl,ic,(3.95-ic)/sl,A.val.min()))
print()
TOPK=12; top=A.nsmallest(TOPK,'val'); rest=A.drop(top.index)
def nrm(s,vals):
    n,k=s[0],s[1]
    if k=='cat': return np.array([(s[2].index(x)+0.5)/len(s[2]) for x in vals])
    if k=='log': return (np.log(vals)-np.log(s[2]))/(np.log(s[3])-np.log(s[2]))
    return (vals-s[2])/(s[3]-s[2])
print('boundary pressure in the NEW box (top %d vs rest):'%TOPK)
still=[]
for s in SPACE:
    a,b=nrm(s,top[s[0]].values),nrm(s,rest[s[0]].values)
    try: pp=mannwhitneyu(a,b).pvalue
    except Exception: pp=1.0
    f=''
    if pp<0.05:
        f='still pinned LOW' if np.mean(a)<0.22 else ('still pinned HIGH' if np.mean(a)>0.78 else 'interior optimum')
        if f.startswith('still'): still.append(s[0])
    print('  %-12s top %.3f  rest %.3f  p %.4f  %s'%(s[0],np.mean(a),np.mean(b),pp,f))
print()
print('parameters still at a wall:',still if still else 'none — the box now contains the optimum')

floor=triv_cap; fin=S2[S2['name']==(win if verdict=='ADOPT' else 'v8_cand5')].iloc[0]
print('\n'+'='*74)
print('%-40s %9s %9s %8s'%('','capped-55','raw','skill'))
for lab,c,r in [('MSTSDN published',3.950,None),
                ('ours, v9 expanded search',fin.capped,fin.raw),
                ('ours, v8 search',5.015,10.131),
                ('ours, v7 matched protocol',5.137,10.268),
                ('trivial floor',floor,triv_raw)]:
    print('%-40s %9.3f %9s %+8.3f'%(lab,c,('%.3f'%r) if r is not None else '  --  ',1-c/floor))
print('='*74)
print('remaining gap to MSTSDN: %+.3f RMSE'%(fin.capped-3.95))
S2.to_csv('ncmapss_v9_stage2.csv',index=False)
pd.DataFrame(TRIALS).to_csv('ncmapss_v9_all_trials.csv',index=False)
json.dump(dict(verdict=verdict,winner=win,spearman=float(rho),slope=float(sl),intercept=float(ic),
               still_pinned=still,final=fin.to_dict(),trivial_capped=floor,trivial_raw=triv_raw),
          open('ncmapss_v9_summary.json','w'),indent=2,default=str)
import os as _os
print('\nsaved to',_os.getcwd())